# 训练 小橘推送 的 4B 模型（工具调用）

用法：
1. 右边 **Settings → Accelerator** 选 **GPU T4 x2**（或 P100），**Internet** 打开。
2. **Add-ons → Secrets** 加一个 `HF_TOKEN`（Hugging Face 的 write 令牌），并勾上让这个笔记本能用。
3. 点上面 **Run All**。大概 1～2 小时，跑完模型会传到 Hugging Face 的 `langhua1998/Qwen3.5-4B-xiaoju-GGUF`，App 里就能下载「4B 小橘版」。


In [ ]:
# 1. 装 Unsloth（Qwen3.5 要 transformers v5）
!pip install -q --upgrade --no-cache-dir unsloth unsloth_zoo
!pip install -q --upgrade "transformers>=5" trl datasets huggingface_hub

In [ ]:
# 2. 拿训练数据：从仓库拉脚本，按 App 里同一份提示词生成 1600 条
BRANCH = "claude/project-thread-3k5i2p"  # PR 合并后也可以改成 "main"
!rm -rf meta-work && git clone -q --depth 1 -b $BRANCH https://github.com/langhua98/meta-work.git
%cd meta-work
!python3 scripts/finetune/make_data.py 1600
%cd ..

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
HF_REPO = "langhua1998/Qwen3.5-4B-xiaoju-GGUF"
OUT_NAME = "Qwen3.5-4B-xiaoju-Q4_K_M.gguf"
MAX_LEN = 3072

In [ ]:
# 3. 加载 4B（16 位 LoRA；官方说 Qwen3.5 不建议 4 位 QLoRA）
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3.5-4B",
    max_seq_length = MAX_LEN,
    load_in_4bit = False,
    load_in_16bit = True,
    full_finetuning = False,
)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    max_seq_length = MAX_LEN,
)

In [ ]:
# 4. 数据：只学 assistant 那段（JSON），系统提示和用户的话不算损失
from datasets import load_dataset
ds = load_dataset("json", data_files={"train": "meta-work/scripts/finetune/data/train.jsonl",
                                      "eval": "meta-work/scripts/finetune/data/eval.jsonl"})
print(ds)
print(ds["train"][0]["text"][-300:])

In [ ]:
# 5. 训练（1 轮就够，数据格式很固定）
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
import torch
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = ds["train"],
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = MAX_LEN,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps = 10,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        output_dir = "outputs",
        report_to = "none",
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n<think>\n\n</think>\n\n",
)
trainer.train()

In [ ]:
# 6. 先导出 Q4_K_M 并传到 Hugging Face（文件名固定，App 按这个下载）
import glob, shutil
from huggingface_hub import HfApi
# 先存一份 LoRA（几十 MB），万一转 GGUF 出错也不白训
model.save_pretrained("lora"); tokenizer.save_pretrained("lora")
model.save_pretrained_gguf("gguf_out", tokenizer, quantization_method = "q4_k_m")
# Unsloth 实际写到 gguf_out_gguf/ 里，整个目录树都找一遍
cands = [f for f in glob.glob("**/*.gguf", recursive=True) if "q4_k_m" in f.lower() and "mmproj" not in f.lower()]
print(cands)
src = max(cands, key=os.path.getsize)
api = HfApi(token = HF_TOKEN)
api.create_repo(HF_REPO, exist_ok = True)
api.upload_file(path_or_fileobj = src, path_in_repo = OUT_NAME, repo_id = HF_REPO)
print("传好了：", f"https://huggingface.co/{HF_REPO}/resolve/main/{OUT_NAME}", os.path.getsize(src))

In [ ]:
# 7. 模型已经导出了，再用留出来的题考一下（考试出错不影响模型）：输出的 JSON 和标准答案的 do 一样就算对
import json
FastLanguageModel.for_inference(model)
ok = 0
rows = ds["eval"]
for r in rows:
    t = r["text"]
    prompt, answer = t.rsplit("</think>\n\n", 1)
    prompt += "</think>\n\n"
    want = json.loads(answer.replace("<|im_end|>\n", ""))
    ids = tokenizer(text=prompt, return_tensors="pt").to("cuda")
    out = model.generate(**ids, max_new_tokens=200, do_sample=False)
    got = tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    try:
        g = json.loads(got[got.index("{"): got.rindex("}") + 1])
        hit = g.get("do") == want.get("do")
    except Exception:
        hit = False
    ok += hit
    if not hit:
        print("✗", prompt.rsplit("<|im_start|>user\n", 1)[1][:60].replace("\n", " "), "→", got[:120], " 期望", want)
print(f"{ok}/{len(rows)} 对")